# Otimização — capítulo 3: Otimização multidimensional sem restrições
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/notebooks/03_multivariate_optimization.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 3 of the course "Optimization" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/optimization-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.2.1 Método de pesquisa aleatória

Exemplo [`ex03_2_1_random_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_1_random_search/ex03_2_1_random_search.py) · funções da UC: [`random_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_1_random_search/random_search.py), [`local_random_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_1_random_search/local_random_search.py)

```text
Reproduz os exemplos do deck 3.2.1 (pesquisa aleatória pura e localizada).

1. f = x1^2 + x2^2 em [-5,5]^2, uma corrida (semente 1): melhor f com
   N = 100 e N = 1000 amostras (slide 1: 1.406 e 0.053).
2. Análise experimental: 30 corridas por N (gerador com semente 7, partilhado
   pelas corridas, como no script das figuras): mediana e quartis (slide 3).
3. Localizada no Rosenbrock de (-1.5, 2), m = 20, r0 = 1, gamma = 0.9:
   uma corrida (semente 3), 150 iterações, n_f = 3001 (slide 5);
   30 corridas (sementes 0-29): n_f até f < 1e-4 = 975 [867; 1052].
4. Pura vs. localizada, orçamento 3000, 30 corridas (slide 6).

Método estocástico: usa o gerador numpy.random.default_rng e as mesmas
sementes dos scripts das figuras e do caderno do capítulo 3, por isso
reproduz EXATAMENTE os números dos slides.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from random_search import random_search
from local_random_search import local_random_search


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


def mq(v):
    """mediana e quartis [Q1, Q3] (percentis 50, 25, 75, como nos scripts)."""
    return np.percentile(v, [50, 25, 75])


ok = True
print("Otimização — deck 3.2.1: pesquisa aleatória pura e localizada")
print("(Python: mesmas sementes e mesmo gerador dos scripts das figuras -> reprodução exata)")

sq = lambda x: x[0]**2 + x[1]**2
ros = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# ------------------------------------------------------------ 1. uma corrida
a, b = [-5, -5], [5, 5]
print("\n1. f = x1^2 + x2^2 em [-5,5]^2, uma corrida (semente 1), N = 1000")
r = random_search(sq, a, b, 1000, s=1, verbose=True)
H = r.history
f100 = H[H[:, 0] <= 100, -1][-1]          # melhor f com as primeiras 100 amostras
print("N = 100: melhor f = %.3f;  N = 1000: melhor f = %.3f em (%.4f, %.4f);  n_f = %d"
      % (f100, r.fx, r.x[0], r.x[1], r.nfev))
c = [confere(f100, 1.406, 3), confere(r.fx, 0.053, 3), r.nfev == 1000]
print("  N = 100, f = 1.406: %s | N = 1000, f = 0.053: %s | n_f = N: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. 30 corridas
print("\n2. Análise experimental: 30 corridas por N (gerador com semente 7)")
g = np.random.default_rng(7)
tab = {}
for N in (10, 30, 100, 300, 1000, 3000, 10000):   # a ordem do script (o gerador é partilhado)
    fb = [random_search(sq, a, b, N, s=g).fx for _ in range(30)]
    tab[N] = mq(fb)
# slide: N, melhor f (mediana), ||x_best|| mediana [Q1; Q3]
T = {10: (2.3, 1, 1.5, 1.2, 2.1, 1), 100: (0.29, 2, 0.54, 0.38, 0.76, 2),
     1000: (0.018, 3, 0.14, 0.09, 0.17, 2), 10000: (0.0028, 4, 0.053, 0.027, 0.070, 3)}
print("%6s %10s %10s %10s %10s" % ("N", "f (med)", "||x|| med", "Q1", "Q3"))
c = []
for N, (fs, df, xm, x1, x3, dx) in T.items():
    med, q1, q3 = tab[N]; nx = np.sqrt([med, q1, q3])   # ||x|| = sqrt(f): comuta com os percentis
    print("%6d %10.4g %10.3g %10.3g %10.3g" % (N, med, *nx))
    c.append(confere(med, fs, df) and confere(nx, [xm, x1, x3], dx))
print("  tabela N = 10, 100, 1000, 10000: %s | %s | %s | %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. localizada
X0 = [-1.5, 2.0]
print("\n3. Localizada no Rosenbrock de (-1.5, 2), m = 20, r0 = 1, gamma = 0.9")
r = local_random_search(ros, X0, 1.0, m=20, gamma=0.9, kmax=150, s=3)
print("uma corrida (semente 3): %d iterações, n_f = %d, x = (%.6f, %.6f), f = %.2e"
      % (r.nit, r.nfev, r.x[0], r.x[1], r.fx))
print("  primeiras e últimas linhas de info.history [k, n_f, r, f(x), x1, x2]:")
for row in np.r_[r.history[:4], r.history[-2:]]:
    print("  %5d %6d %10.3e %12.4e %10.6f %10.6f" % tuple(row))
c = [r.nit == 150, r.nfev == 3001, confere(r.x, [1, 1], 4), confere(r.fx / 1e-13, 5.9, 1)]
print("  150 it.: %s | n_f = 3001: %s | x -> (1,1): %s | f = 5.9e-13: %s" % simnao(c))
ok = ok and all(c)

nhit = [local_random_search(ros, X0, 1.0, m=20, gamma=0.9, tolx=1e-8, kmax=5000,
                            ftarget=1e-4, s=s).nhit for s in range(30)]
med, q1, q3 = mq(nhit)
print("30 corridas (sementes 0-29): n_f até f < 1e-4 = %.0f [%.0f; %.0f] (mediana [Q1; Q3]); falhas: %d"
      % (med, q1, q3, int(np.sum(np.isinf(nhit)))))
c = [confere([med, q1, q3], [975, 867, 1052], 0)]
print("  975 [867; 1052]: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 4. pura vs. localizada
print("\n4. Pura vs. localizada no Rosenbrock, orçamento 3000, 30 corridas (sementes 0-29)")
fp = [random_search(ros, [-2, -1], [2, 3], 3000, s=s).fx for s in range(30)]
fl = [local_random_search(ros, X0, 1.0, m=20, gamma=0.9, kmax=150, s=s).fx for s in range(30)]
mp, ml = mq(fp), mq(fl)
print("pura em [-2,2]x[-1,3], n_f = 3000:  melhor f = %.3f [%.3f; %.3f]" % tuple(mp))
print("localizada, 150 it., n_f = 3001:    melhor f = %.1e [%.1e; %.1e]" % tuple(ml))
c = [confere(mp, [0.014, 0.008, 0.022], 3), confere(ml / [1e-12, 1e-13, 1e-12], [2, 5, 8], 0)]
print("  pura 0.014 [0.008; 0.022]: %s | localizada 2e-12 [5e-13; 8e-12]: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.1: pesquisa aleatória pura e localizada
(Python: mesmas sementes e mesmo gerador dos scripts das figuras -> reprodução exata)

1. f = x1^2 + x2^2 em [-5,5]^2, uma corrida (semente 1), N = 1000
     i         x1         x2     f_best
     1     0.1182     4.5046    20.3057
     3    -1.8817    -0.7667     4.1286
     8    -1.9681    -0.4650     4.0895
    51     1.5387    -0.6877     2.8405
    59    -1.2376    -0.7908     2.1570
    63    -0.5160    -1.3230     2.0166
    88    -1.1851     0.0380     1.4058
   108     0.2922     1.1158     1.3304
   152     0.5741    -0.0101     0.3297
   173    -0.0599     0.2222     0.0530
N = 100: melhor f = 1.406;  N = 1000: melhor f = 0.053 em (-0.0599, 0.2222);  n_f = 1000
  N = 100, f = 1.406: sim | N = 1000, f = 0.053: sim | n_f = N: sim

2. Análise experimental: 30 corridas por N (gerador com semente 7)


     N    f (med)  ||x|| med         Q1         Q3
    10      2.305       1.52       1.21       2.07
   100     0.2886      0.537      0.379      0.764
  1000    0.01842      0.136     0.0929      0.166
 10000   0.002764     0.0526     0.0269     0.0699
  tabela N = 10, 100, 1000, 10000: sim | sim | sim | sim

3. Localizada no Rosenbrock de (-1.5, 2), m = 20, r0 = 1, gamma = 0.9
uma corrida (semente 3): 150 iterações, n_f = 3001, x = (1.000000, 1.000000), f = 5.88e-13
  primeiras e últimas linhas de info.history [k, n_f, r, f(x), x1, x2]:
      0      1  1.000e+00   1.2500e+01  -1.500000   2.000000
      1     21  1.000e+00   6.8370e+00  -1.030846   1.227344
      2     41  1.000e+00   3.3334e+00  -0.805027   0.620623
      3     61  9.000e-01   3.3334e+00  -0.805027   0.620623
    149   2981  2.616e-06   5.8795e-13   1.000000   1.000000
    150   3001  2.354e-06   5.8795e-13   1.000000   1.000000
  150 it.: sim | n_f = 3001: sim | x -> (1,1): sim | f = 5.9e-13: sim


30 corridas (sementes 0-29): n_f até f < 1e-4 = 975 [867; 1052] (mediana [Q1; Q3]); falhas: 0
  975 [867; 1052]: sim

4. Pura vs. localizada no Rosenbrock, orçamento 3000, 30 corridas (sementes 0-29)


pura em [-2,2]x[-1,3], n_f = 3000:  melhor f = 0.014 [0.008; 0.022]
localizada, 150 it., n_f = 3001:    melhor f = 1.9e-12 [5.3e-13; 7.7e-12]
  pura 0.014 [0.008; 0.022]: sim | localizada 2e-12 [5e-13; 8e-12]: sim

confere com os slides: sim


## 3.2.2 Pesquisa em grelha

Exemplo [`ex03_2_2_grid_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_2_grid_search/ex03_2_2_grid_search.py) · funções da UC: [`grid_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_2_grid_search/grid_search.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 3.2.2 (pesquisa em grelha).

Himmelblau em [-5,5]^2:
1. Grelha 10 x 10 (100 avaliações): melhor f = 4.51; os 4 melhores nós,
   f = 4.51, 4.70, 6.15, 11.88, ficam um em cada bacia.
2. Grosseira -> fina: mais 100 avaliações (grelha 10 x 10 em [x - h, x + h],
   h = 10/9) à volta do melhor nó: f = 0.32.
3. Grelha + Nelder–Mead (3.2.3) a partir de cada um dos 4 melhores nós
   (simplex do deck, tolerâncias 1e-6): n_f = 89, 94, 94, 87; os 4 mínimos
   globais, todos com f < 1e-11, em 464 avaliações no total; o 5.º melhor
   nó leva outra vez a (3,2), com mais 95 avaliações.

Usa nelder_mead da pasta 03_2_3_nelder_mead (posta no sys.path por
uc_setup; não há cópia do módulo nesta pasta).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [3]:
import numpy as np

from grid_search import grid_search
from nelder_mead import nelder_mead  # (3.2.3)


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.2.2: pesquisa em grelha")

himmel = lambda x: (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
MIN = np.array([[3, 2], [-2.805118, 3.131312], [-3.779310, -3.283186], [3.584428, -1.848126]])
bacia = lambda p: int(np.argmin(np.linalg.norm(MIN - p, axis=1))) + 1   # mínimo mais próximo

# ------------------------------------------------------------ 1. grelha 10 x 10
print("\n1. Himmelblau em [-5,5]^2, grelha 10 x 10")
r = grid_search(himmel, [-5, -5], [5, 5], 10, verbose=True)
top4 = r.history[:4]
b4 = [bacia(p) for p in top4[:, :2]]
print("melhor nó (%.4f, %.4f), f = %.2f;  n_f = %d;  bacias dos 4 melhores: %d %d %d %d"
      % (*r.x, r.fx, r.nfev, *b4))
c = [confere(r.fx, 4.51, 2), r.nfev == 100, confere(top4[:, 2], [4.51, 4.70, 6.15, 11.88], 2),
     len(set(b4)) == 4]
print("  f = 4.51: %s | n_f = 100: %s | 4 melhores: %s | um em cada bacia: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. grosseira -> fina
h = r.h
r2 = grid_search(himmel, r.x - h, r.x + h, 10)
print("\n2. Grosseira -> fina: grelha 10 x 10 em [x - h, x + h], h = 10/9 = %.4f" % h[0])
print("melhor nó (%.4f, %.4f), f = %.2f;  mais %d avaliações" % (*r2.x, r2.fx, r2.nfev))
c = [confere(r2.fx, 0.32, 2), r2.nfev == 100]
print("  f = 0.32: %s | +100 avaliações: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. grelha + Nelder–Mead
print("\n3. Grelha + Nelder–Mead a partir dos 5 melhores nós (tolx = tolf = 1e-6)")
print("  nó      ponto inicial         f                 mínimo    f final   n_f")
tot = r.nfev; nf = []; ff = []; bm = []
for i in range(5):
    p = r.history[i, :2]
    rm = nelder_mead(himmel, p, tolx=1e-6, tolf=1e-6)   # simplex do deck a partir de p
    nf.append(rm.nfev); ff.append(rm.fx); bm.append(bacia(rm.x))
    print("%3d  (%7.4f, %7.4f) %9.2f  (%9.6f, %9.6f) %10.1e %5d"
          % (i + 1, *p, r.history[i, 2], *rm.x, rm.fx, rm.nfev))
    if i < 4:
        tot += rm.nfev
print("total grelha + 4 Nelder–Mead: %d avaliações;  o 5.º nó leva ao mínimo %d, com mais %d"
      % (tot, bm[4], nf[4]))
c = [nf[:4] == [89, 94, 94, 87], all(v < 1e-11 for v in ff[:4]), len(set(bm[:4])) == 4,
     tot == 464, bm[4] == 1, nf[4] == 95]
print("  n_f = 89, 94, 94, 87: %s | f < 1e-11: %s | 4 mínimos diferentes: %s | total 464: %s\n"
      "  5.º nó -> (3,2): %s | mais 95: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.2: pesquisa em grelha

1. Himmelblau em [-5,5]^2, grelha 10 x 10
   i         x1         x2            f
   1    -2.7778     2.7778       4.5069
   2     2.7778     1.6667       4.7020
   3     3.8889    -1.6667       6.1469
   4    -3.8889    -2.7778      11.8778
   5     2.7778     2.7778      12.4630
melhor nó (-2.7778, 2.7778), f = 4.51;  n_f = 100;  bacias dos 4 melhores: 2 1 4 3
  f = 4.51: sim | n_f = 100: sim | 4 melhores: sim | um em cada bacia: sim

2. Grosseira -> fina: grelha 10 x 10 em [x - h, x + h], h = 10/9 = 1.1111
melhor nó (-2.9012, 3.1481), f = 0.32;  mais 100 avaliações
  f = 0.32: sim | +100 avaliações: sim

3. Grelha + Nelder–Mead a partir dos 5 melhores nós (tolx = tolf = 1e-6)
  nó      ponto inicial         f                 mínimo    f final   n_f
  1  (-2.7778,  2.7778)      4.51  (-2.805118,  3.131312)    4.2e-12    89
  2  ( 2.7778,  1.6667)      4.70  ( 3.000000,  2.000000)    1.3e-12    94
  3  ( 3.8889, -1.6667)      6.15  ( 3.5844

## 3.2.3 Método do simplex de Nelder–Mead

Exemplo [`ex03_2_3_nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/ex03_2_3_nelder_mead.py) · funções da UC: [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 3.2.3 (Nelder–Mead).

1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3).
2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2),
   tolerâncias 1e-6: tabela k = 0..12; 54 iterações, n_f = 106, x* = (3,2).
   Com o simplex (0,0), (-1.2,0), (0,-1.2) chega-se a (-2.81; 3.13).
3. Rosenbrock a partir de (-1.5, 2): n_f até f < 1e-4 = 166 (simplex de
   arestas 0.5, o da comparação final em 3.3.3) e 139 (arestas de 5 %,
   o simplex inicial do deck).

O deck usa fminsearch (MATLAB) / scipy.optimize.minimize (Python); aqui
usa-se a implementação da UC, nelder_mead, com o pseudocódigo do deck.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [4]:
import numpy as np

from nelder_mead import nelder_mead


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.2.3: método de Nelder–Mead")

sq = lambda x: x[0]**2 + x[1]**2
himmel = lambda x: (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
ros = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# ------------------------------------------------------------ 1. à mão
print("\n1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3)")
r = nelder_mead(sq, [[-1, 0], [1, 2], [-1, 3]], kmax=1, verbose=True)
S = r.simplex; Fs = [sq(x) for x in S]
print("novo simplex: (%g,%g), (%g,%g), (%g,%g) com f = %g, %g, %g;  avaliações nesta iteração: %d"
      % (*S[0], *S[1], *S[2], *Fs, r.nfev - 3))
c = [r.ops[1] == "reflexão", confere(S, [[-1, 0], [1, -1], [1, 2]], 4),
     confere(Fs, [1, 2, 5], 4), r.nfev - 3 == 1]
print("  reflexão aceite: %s | novo simplex: %s | f = 1, 2, 5: %s | 1 avaliação: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. Himmelblau
print("\n2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2), tolx = tolf = 1e-6")
r = nelder_mead(himmel, [[0, 0], [1.2, 0], [0, 1.2]], tolx=1e-6, tolf=1e-6)
print("primeiras 13 linhas de info.history [k, n_f, f(x_b), f(x_w), x_b] e operação:")
print("%4s %5s %12s %12s %10s %10s  %s" % (*r.cols, "operação"))
for row, o in zip(r.history[:13], r.ops[:13]):
    print("%4d %5d %12.4f %12.4f %10.4f %10.4f  %s" % (*row, o))
print("... %d iterações, n_f = %d, x* = (%.6f, %.6f), f = %.1e  (%s)"
      % (r.nit, r.nfev, r.x[0], r.x[1], r.fx, r.message))
# tabela do slide (k = 0..12): operação, x_b, f(x_b), f(x_w)
Tops = ["inicial", "expansão", "reflexão", "reflexão", "contr. int.", "contr. int.", "contr. ext.",
        "contr. int.", "contr. int.", "contr. int.", "contr. int.", "contr. ext.", "contr. int."]
Txb = [[1.200, 0.000], [1.800, 1.800], [3.000, 0.600], [3.000, 0.600], [2.550, 1.650],
       [3.188, 1.762], [3.188, 1.762], [3.188, 1.762], [2.884, 1.921], [2.919, 2.050],
       [2.919, 2.050], [3.031, 1.983], [3.031, 1.983]]
Tfb = [125.0336, 39.3632, 15.2096, 15.2096, 11.0925, 1.3499, 1.3499, 1.3499, 0.7628,
       0.1972, 0.1972, 0.0303, 0.0303]
Tfw = [170.000, 126.954, 125.034, 39.363, 24.579, 15.210, 11.093, 2.965, 1.604, 1.350,
       0.763, 0.217, 0.197]
H = r.history[:13]
c = [r.ops[:13] == Tops, confere(H[:, 4:6], Txb, 3), confere(H[:, 2], Tfb, 4),
     confere(H[:, 3], Tfw, 3), r.nit == 54, r.nfev == 106, confere(r.x, [3, 2], 4), r.fx < 1e-10]
print("  tabela k = 0..12: operação: %s | x_b: %s | f(x_b): %s | f(x_w): %s\n"
      "  54 it.: %s | n_f = 106: %s | x* = (3,2): %s | f ~ 0: %s" % simnao(c))
ok = ok and all(c)

r2 = nelder_mead(himmel, [[0, 0], [-1.2, 0], [0, -1.2]])
print("simplex (0,0), (-1.2,0), (0,-1.2): x* = (%.2f; %.2f), f = %.1e" % (r2.x[0], r2.x[1], r2.fx))
c = [confere(r2.x, [-2.81, 3.13], 2)]
print("  chega a (-2.81; 3.13): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. Rosenbrock
x0 = np.array([-1.5, 2.0])
print("\n3. Rosenbrock a partir de (-1.5, 2): n_f até f < 1e-4 (info.nhit)")
ra = nelder_mead(ros, [x0, x0 + [0.5, 0], x0 + [0, 0.5]], ftarget=1e-4)
rb = nelder_mead(ros, x0, ftarget=1e-4)          # simplex do deck: h_i = 0.05 max(1,|x0_i|)
print("simplex de arestas 0.5:     n_f até f < 1e-4 = %d  (final: %d it., n_f = %d, x = (%.4f, %.4f))"
      % (ra.nhit, ra.nit, ra.nfev, *ra.x))
print("simplex de arestas de 5 %%:  n_f até f < 1e-4 = %d  (final: %d it., n_f = %d, x = (%.4f, %.4f))"
      % (rb.nhit, rb.nit, rb.nfev, *rb.x))
c = [ra.nhit == 166, rb.nhit == 139]
print("  n_f = 166: %s | n_f = 139: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.3: método de Nelder–Mead

1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3)
   k   n_f       f(x_b)       f(x_w)       x_b1       x_b2  operação
   0     3       1.0000      10.0000    -1.0000     0.0000  inicial
   1     4       1.0000       5.0000    -1.0000     0.0000  reflexão
novo simplex: (-1,0), (1,-1), (1,2) com f = 1, 2, 5;  avaliações nesta iteração: 1
  reflexão aceite: sim | novo simplex: sim | f = 1, 2, 5: sim | 1 avaliação: sim

2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2), tolx = tolf = 1e-6
primeiras 13 linhas de info.history [k, n_f, f(x_b), f(x_w), x_b] e operação:
   k   n_f       f(x_b)       f(x_w)       x_b1       x_b2  operação
   0     3     125.0336     170.0000     1.2000     0.0000  inicial
   1     5      39.3632     126.9536     1.8000     1.8000  expansão
   2     7      15.2096     125.0336     3.0000     0.6000  reflexão
   3     8      15.2096      39.3632     3.0000     0.6000  reflexão
   4

## 3.2.4 Método de Box — versão simplificada de EVOP

Exemplo [`ex03_2_4_box.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_4_box/ex03_2_4_box.py) · funções da UC: [`box_evo.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_4_box/box_evo.py)

```text
Reproduz os exemplos do deck 3.2.4 (método de Box, EVOP simplificado).

«Uma iteração à mão»: Himmelblau em x = (0,0), Delta = (2,2): f nos 4 vértices.
Himmelblau a partir de (0,0), Delta = (2,2), tolx = 1e-4: tabela k = 0..9,
  20 iterações, n_f = 4*20 + 1 = 81, termina em (3,2) com f = 0.
Rosenbrock a partir de (-1.5, 2), Delta_0 = (1,1): n_f = 13 445 até f < 1e-4
  (tolx = 1e-6, como no caderno cap3_comparacao.ipynb; o deck não indica tolx);
  a caixa tem de encolher até Delta ~ 6e-5.
«Porque falha em vales curvos?»: em (0,0), passo 0.25, as 4 diagonais dão
  f entre 4.08 e 11.33 (> 1), mas f(0.25, 0) = 0.953.

Contagens: f(x0) conta e em cada iteração avaliam-se SEMPRE os 2^n vértices
(nota do deck: «nas contagens deste deck avaliam-se sempre os 2^n vértices»).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [5]:
import numpy as np

from box_evo import box_evo


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
himmel = lambda x: (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

print("Otimização — deck 3.2.4: método de Box (EVOP simplificado)")

# ------------------------------------------------------------ Himmelblau
x0, Delta, tolx = [0, 0], [2, 2], 1e-4
print("\nHimmelblau a partir de (%g,%g), Delta = (%g,%g), tolx = %g" % (*x0, *Delta, tolx))
r = box_evo(himmel, x0, Delta, tolx, verbose=True)
print(r.message)
print("x = (%.4f, %.4f), f(x) = %.4g;  %d iterações, n_f = %d (= 4 x %d + 1)"
      % (*r.x, r.fx, r.nit, r.nfev, r.nit))

# «Uma iteração à mão»: vértices (-1,-1), (1,-1), (-1,1), (1,1), pela ordem do slide
# (a ordem de avaliação de box_evo é (-1,-1), (-1,1), (1,-1), (1,1))
fv = r.ftrace[[1, 3, 2, 4]]
print("Iteração à mão: f(0,0) = %g; vértices (-1,-1), (1,-1), (-1,1), (1,1): %g, %g, %g, %g"
      % (r.ftrace[0], *fv))

# tabela do slide (k = 0..9): x_k(1) x_k(2)  Delta  f(x_k)  decisão (1 = move)  f novo
T = [[0.0, 0.0, 2.000, 170.000, 1, 106.000],
     [1.0, 1.0, 2.000, 106.000, 1, 26.000],
     [2.0, 2.0, 2.000, 26.000, 1, 10.000],
     [3.0, 1.0, 2.000, 10.000, 0, 10.000],
     [3.0, 1.0, 1.000, 10.000, 1, 9.125],
     [3.5, 1.5, 1.000, 9.125, 1, 0.000],
     [3.0, 2.0, 1.000, 0.000, 0, 0.000],
     [3.0, 2.0, 0.500, 0.000, 0, 0.000],
     [3.0, 2.0, 0.250, 0.000, 0, 0.000],
     [3.0, 2.0, 0.125, 0.000, 0, 0.000]]
H = r.history[:10, [1, 2, 3, 5, 6, 7]]
c = [confere(r.ftrace[0], 170, 0), confere(fv, [170, 146, 130, 106], 0),
     confere(H, T, 3), r.nit == 20, r.nfev == 81,
     confere(r.x, [3, 2], 3), confere(r.fx, 0, 3)]
print("  f(0,0) = 170: %s | vértices 170/146/130/106: %s | tabela k = 0..9: %s | "
      "20 it.: %s | n_f = 81: %s | x = (3,2): %s | f = 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ Rosenbrock
x0, Delta, tolx = [-1.5, 2], [1, 1], 1e-6
print("\nRosenbrock a partir de (%g,%g), Delta_0 = (%g,%g), tolx = %g" % (*x0, *Delta, tolx))
r = box_evo(rosen, x0, Delta, tolx)
print(r.message)
hit = int(np.argmax(r.ftrace < 1e-4)) + 1
kh = int(np.ceil((hit - 1) / 4))       # linha de history (iteração k = kh - 1), a contar de 1
Dh = r.history[kh - 1, 3]              # Delta_1 nessa iteração
print("primeira avaliação com f < 1e-4: n_f = %d (na iteração k = %d, com Delta = %.2e)"
      % (hit, kh - 1, Dh))
print("no fim: x = (%.6f, %.6f), f = %.2e;  %d iterações, n_f total = %d"
      % (*r.x, r.fx, r.nit, r.nfev))
c = [hit == 13445, confere(Dh, 6e-5, 5)]
print("  n_f = 13 445 até f < 1e-4: %s | Delta ~ 6e-5: %s" % simnao(c))
ok = ok and all(c)

# --------------------------------------- «Porque falha em vales curvos?»
r = box_evo(rosen, [0, 0], [0.5, 0.5], 0, kmax=1)       # uma só iteração
fd = r.ftrace[1:5]; fe1 = rosen([0.25, 0])
nomes = ("encolhe", "move")
print("\nRosenbrock em (0,0), f = %g, passo 0.25: diagonais f = %s -> %s"
      % (r.ftrace[0], "".join("%.2f " % v for v in fd), nomes[int(r.history[0, 6])]))
print("eixo +e1: f(0.25, 0) = %.3f < 1" % fe1)
c = [bool(np.all(fd > 1)), confere(fd.min(), 4.08, 2), confere(fd.max(), 11.33, 2),
     r.history[0, 6] == 0, confere(fe1, 0.953, 3)]
print("  nenhuma diagonal melhora: %s | 4.08: %s | 11.33: %s | encolhe: %s | 0.953: %s"
      % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.4: método de Box (EVOP simplificado)

Himmelblau a partir de (0,0), Delta = (2,2), tolx = 0.0001
    k        x1        x2    Delta1    Delta2      f(x_k)   decisão       f_novo     n_f
    0    0.0000    0.0000    2.0000    2.0000    170.0000      move     106.0000       5
    1    1.0000    1.0000    2.0000    2.0000    106.0000      move      26.0000       9
    2    2.0000    2.0000    2.0000    2.0000     26.0000      move      10.0000      13
    3    3.0000    1.0000    2.0000    2.0000     10.0000   encolhe      10.0000      17
    4    3.0000    1.0000    1.0000    1.0000     10.0000      move       9.1250      21
    5    3.5000    1.5000    1.0000    1.0000      9.1250      move       0.0000      25
    6    3.0000    2.0000    1.0000    1.0000      0.0000   encolhe       0.0000      29
    7    3.0000    2.0000    0.5000    0.5000      0.0000   encolhe       0.0000      33
    8    3.0000    2.0000    0.2500    0.2500      0.0000   encolhe       0.0000

norm(Delta) = 6.74e-07 < tolx = 1e-06 ao fim de 6474 iterações
primeira avaliação com f < 1e-4: n_f = 13445 (na iteração k = 3360, com Delta = 6.10e-05)
no fim: x = (0.999880, 0.999759), f = 1.45e-08;  6474 iterações, n_f total = 25897
  n_f = 13 445 até f < 1e-4: sim | Delta ~ 6e-5: sim

Rosenbrock em (0,0), f = 1, passo 0.25: diagonais f = 11.33 5.08 10.33 4.08  -> encolhe
eixo +e1: f(0.25, 0) = 0.953 < 1
  nenhuma diagonal melhora: sim | 4.08: sim | 11.33: sim | encolhe: sim | 0.953: sim

confere com os slides: sim


## 3.2.5 Método de Hooke–Jeeves

Exemplo [`ex03_2_5_hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/ex03_2_5_hooke_jeeves.py) · funções da UC: [`exploratory.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/exploratory.py), [`hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/hooke_jeeves.py)

```text
Reproduz os exemplos do deck 3.2.5 (Hooke–Jeeves).

f(x) = 3 x1^2 + x2^2 - 12 x1 - 8 x2  (mínimo em (2,4), f = -28):
  «Uma exploração à mão»: de xb = (1,1), f = -16, com P = (0.5, 0.5):
    x1: (1.5,1) -> -18.25, (0.5,1) -> -12.25: fica (1.5,1);
    x2: (1.5,1.5) -> -21, (1.5,0.5) -> -15: fica (1.5,1.5); 4 avaliações;
  «Exemplo completo»: x0 = (1,1), a = 2, P0 = (0.5,0.5), T = (0.1,0.1):
    padrão (2,2) -> (2,2.5), aceite; (2.5,3.5) -> (2,4), aceite;
    (2,5.5) -> (2,5), rejeitado; exploração em (2,4) falha com P = 0.5,
    0.25, 0.125; 0.0625 < T: para em (2,4), f = -28.
Rosenbrock a partir de (-1.5, 2), P0 = 0.5, a = 2, T = 1e-6 (como no
  caderno cap3_comparacao.ipynb): n_f = 430 até f < 1e-4.
Custo: 2n por exploração (avaliam-se +P_j e -P_j; Deb, 2012); 1 + 2n por
  movimento de padrão.

Contagens: f(x0) conta; f(xb) nunca é reavaliado (fica guardado).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [6]:
import numpy as np

from exploratory import exploratory
from hooke_jeeves import hooke_jeeves


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
quad = lambda x: 3 * x[0]**2 + x[1]**2 - 12 * x[0] - 8 * x[1]
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

print("Otimização — deck 3.2.5: método de Hooke–Jeeves")

# --------------------------------------------- «Uma exploração à mão»
xb = np.array([1.0, 1.0]); fb = quad(xb); P = [0.5, 0.5]
print("\nExploração à mão: xb = (%g,%g), f(xb) = %g, P = (%g,%g)" % (*xb, fb, *P))
r = exploratory(quad, xb, fb, P)
for h in r.history:
    print("  x%d %s P%d: (%g, %g) -> f = %g, %s"
          % (h[0], "+" if h[1] > 0 else "-", h[0], h[2], h[3], h[4],
             "a melhor: guarda-se" if h[5] else "não é a melhor"))
print("xe = (%g, %g), f = %g, com %d avaliações" % (*r.x, r.fx, r.nfev))
c = [confere(fb, -16, 0),
     confere(r.history[:, 2:5], [[1.5, 1, -18.25], [0.5, 1, -12.25], [1.5, 1.5, -21], [1.5, 0.5, -15]], 2),
     confere(r.x, [1.5, 1.5], 1), r.nfev == 4]
print("  f(1,1) = -16: %s | -18.25, -12.25, -21, -15: %s | xe = (1.5,1.5): %s | 4 avaliações: %s" % simnao(c))
ok = ok and all(c)

# --------------------------------------------- «Exemplo completo»
x0, a, P0, T = [1, 1], 2, [0.5, 0.5], [0.1, 0.1]
print("\nExemplo completo: x0 = (%g,%g), a = %g, P0 = (%g,%g), T = (%g,%g)" % (*x0, a, *P0, *T))
r = hooke_jeeves(quad, x0, a, P0, T, verbose=True)
print(r.message)
print("solução x = (%g, %g), f = %g;  n_f = %d" % (*r.x, r.fx, r.nfev))
H = r.history
# linhas 2 a 4 do slide (padrão): xt(1) xt(2) f(xt)  xe'(1) xe'(2) f(xe')  aceite
Tpad = [[2.0, 2.0, -24.00, 2.0, 2.5, -25.75, 1],
        [2.5, 3.5, -27.00, 2.0, 4.0, -28.00, 1],
        [2.0, 5.5, -25.75, 2.0, 5.0, -27.00, 0]]
c = [r.nit == 7 and list(H[:, 1]) == [0, 1, 1, 1, 0, 0, 0],   # explor., 3 padrões, 3 explor.
     confere(H[0, 7:10], [1.5, 1.5, -21], 2),
     confere(H[1:4][:, [4, 5, 6, 7, 8, 9, 13]], Tpad, 2),
     confere(H[4:7, 11], [0.5, 0.25, 0.125], 4) and bool(np.all(H[4:7, 13] == 0)),
     confere(r.P, [0.0625, 0.0625], 4),
     confere(r.x, [2, 4], 4) and confere(r.fx, -28, 4)]
print("  sequência de movimentos: %s | 1.ª exploração -> (1.5,1.5): %s | padrões (2,2), (2.5,3.5), (2,5.5): %s |\n"
      "  falha com P = 0.5, 0.25, 0.125: %s | P = 0.0625 < T: %s | (2,4), f = -28: %s" % simnao(c))
ok = ok and all(c)

# --------------------------------------------- Rosenbrock
x0, a, P0, T = [-1.5, 2], 2, [0.5, 0.5], [1e-6, 1e-6]
print("\nRosenbrock a partir de (%g,%g), a = %g, P0 = (%g,%g), T = (%g,%g)" % (*x0, a, *P0, *T))
r = hooke_jeeves(rosen, x0, a, P0, T)
print(r.message)
hit = int(np.argmax(r.ftrace < 1e-4)) + 1
print("primeira avaliação com f < 1e-4: n_f = %d" % hit)
print("no fim: x = (%.6f, %.6f), f = %.2e;  %d movimentos, n_f total = %d"
      % (*r.x, r.fx, r.nit, r.nfev))
# custo de cada movimento: 2n (exploração), 1 + 2n (padrão), n = 2
H = r.history; n = 2
dn = np.diff(np.concatenate(([1], H[:, -1])))
ce = dn[H[:, 1] == 0]; cp = dn[H[:, 1] == 1]
print("custo por exploração: %d a %d; por movimento de padrão: %d a %d"
      % (ce.min(), ce.max(), cp.min(), cp.max()))
r5 = hooke_jeeves(rosen, x0, a, P0, [1e-5, 1e-5])
print("(com T = 1e-5: n_f = %d até f < 1e-4 e n_f total = %d)"
      % (int(np.argmax(r5.ftrace < 1e-4)) + 1, r5.nfev))
c = [hit == 430, bool(np.all(ce == 2 * n)), bool(np.all(cp == 1 + 2 * n))]
print("  n_f = 430 até f < 1e-4: %s | n_f = 2n por exploração: %s | 1 + 2n por padrão: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.5: método de Hooke–Jeeves

Exploração à mão: xb = (1,1), f(xb) = -16, P = (0.5,0.5)
  x1 + P1: (1.5, 1) -> f = -18.25, a melhor: guarda-se
  x1 - P1: (0.5, 1) -> f = -12.25, não é a melhor
  x2 + P2: (1.5, 1.5) -> f = -21, a melhor: guarda-se
  x2 - P2: (1.5, 0.5) -> f = -15, não é a melhor
xe = (1.5, 1.5), f = -21, com 4 avaliações
  f(1,1) = -16: sim | -18.25, -12.25, -21, -15: sim | xe = (1.5,1.5): sim | 4 avaliações: sim

Exemplo completo: x0 = (1,1), a = 2, P0 = (0.5,0.5), T = (0.1,0.1)
   m movimento      xb1      xb2      xt1      xt2     f(xt)      xe1      xe2     f(xe)       P1       P2  resultado    n_f
   1 explor.     1.0000   1.0000        -        -         -   1.5000   1.5000  -21.0000   0.5000   0.5000  melhora        5
   2 padrão      1.0000   1.0000   2.0000   2.0000  -24.0000   2.0000   2.5000  -25.7500   0.5000   0.5000  aceite        10
   3 padrão      1.5000   1.5000   2.5000   3.5000  -27.0000   2.0000   4.0000  -28.0000   0.5000   0.5000

## 3.3.1 Método do gradiente

Exemplo [`ex03_3_1_steepest_descent.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_1_steepest_descent/ex03_3_1_steepest_descent.py) · funções da UC: [`grad_fd.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_1_steepest_descent/grad_fd.py), [`steepest_descent.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_1_steepest_descent/steepest_descent.py)

```text
Reproduz os exemplos do deck 3.3.1 (método do gradiente) e o do gradiente
numérico do deck 3.3.

«Uma iteração à mão»: f(x) = (x1^2 + 9 x2^2)/2, x0 = (9, 1): alpha_0 = 0.2,
    x1 = (7.2; -0.8), f = 28.8, g1 = (7.2; -7.2), g1^T d0 = 0.
«O percurso em ziguezague»: tabela k = 0..8; 74 iterações para ||g|| < 1e-6,
    n_g = 75; f multiplica por 0.64 em cada iteração.
«No computador»: pesquisa em linha de alta precisão (a das figuras, cerca de
    24 avaliações de f por pesquisa) e do tipo fminbnd (6 por pesquisa): as
    mesmas 74 iterações.
Deck 3.3: gradiente de Rosenbrock em (-1.5; 2) = (-155, -50); com h = 1e-4
    o erro das diferenças centrais é 6e-6.
«Rosenbrock»: de (-1.5; 2), 3000 iterações, ainda f = 2.4e-4, x ~ (0.98; 0.97).

Imprime as tabelas, os valores finais e as contagens, e no fim compara com
os slides (vírgula decimal nos slides, ponto aqui).
```

In [7]:
import numpy as np

from grad_fd import grad_fd
from steepest_descent import steepest_descent


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.3.1: método do gradiente")

f = lambda x: 0.5 * (x[0]**2 + 9 * x[1]**2)       # = x^T A x / 2, A = diag(1, 9)
grad = lambda x: np.array([x[0], 9 * x[1]])
x0 = np.array([9.0, 1.0])

# ------------------------------------------------ uma iteração à mão
g0 = grad(x0); d0 = -g0
r1 = steepest_descent(f, grad, x0, 1e-6, 1)          # uma só iteração
x1 = r1.x; g1 = grad(x1); alpha0 = r1.history[1, 5]
print("\nUma iteração à mão: g0 = (%g, %g), d0 = (%g, %g)" % (*g0, *d0))
print("alpha_0 = %.4f (exato: 162/810 = 0.2), x1 = (%.4f; %.4f), f(x1) = %.4f (era %g)"
      % (alpha0, *x1, f(x1), f(x0)))
print("g1 = (%.4f; %.4f), g1^T d0 = %.1e (ortogonais)" % (*g1, g1 @ d0))
c = [confere(alpha0, 0.2, 4), confere(x1, [7.2, -0.8], 4), confere(f(x1), 28.8, 4),
     confere(g1, [7.2, -7.2], 4), abs(g1 @ d0) < 1e-6]
print("  alpha_0 = 0.2: %s | x1: %s | f = 28.8: %s | g1: %s | g1^T d0 = 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ percurso em ziguezague
tolg = 1e-6
print("\nZiguezague: f(x) = (x1^2 + 9 x2^2)/2, x0 = (9, 1), tolg = %g, pesquisa em linha de Brent (tol 1e-10)" % tolg)
r = steepest_descent(f, grad, x0, tolg, 1000)
H = r.history
print("%4s %22s %12s %10s %9s" % ("k", "x_k", "f(x_k)", "||g_k||", "alpha_k-1"))
for row in H[:9]:
    print("%4d  (%9.4f; %9.4f) %12.5f %10.4f %9.4f" % tuple(row))
print("...")
print("%4d  (%9.2e; %9.2e) %12.2e %10.2e %9.4f" % tuple(H[-1]))
razao = H[1:, 3] / H[:-1, 3]
print("%s: %d iterações, n_g = %d, n_f = %d (%d nas pesquisas em linha + %d iterandos)"
      % (r.message, r.nit, r.ngev, r.nfev, r.nfev_ls, r.nit + 1))
print("média de %.1f avaliações de f por pesquisa em linha (o slide diz «cerca de 24»)" % (r.nfev_ls / r.nit))
print("f_{k+1}/f_k: mín. %.4f, máx. %.4f (cota de Kantorovich ((9-1)/(9+1))^2 = 0.64)" % (razao.min(), razao.max()))
T = np.array([[9.0000, 1.0000, 45.00000, 12.7279, np.nan],
              [7.2000, -0.8000, 28.80000, 10.1823, 0.2000],
              [5.7600, 0.6400, 18.43200, 8.1459, 0.2000],
              [4.6080, -0.5120, 11.79648, 6.5167, 0.2000],
              [3.6864, 0.4096, 7.54975, 5.2134, 0.2000],
              [2.9491, -0.3277, 4.83184, 4.1707, 0.2000],
              [2.3593, 0.2621, 3.09238, 3.3365, 0.2000],
              [1.8874, -0.2097, 1.97912, 2.6692, 0.2000],
              [1.5099, 0.1678, 1.26664, 2.1354, 0.2000]])
c = [confere(H[:9, 1:3], T[:, 0:2], 4) and confere(H[:9, 3], T[:, 2], 5) and confere(H[:9, 4], T[:, 3], 4)
     and confere(H[1:9, 5], T[1:, 4], 4),
     r.nit == 74, r.ngev == 75, r.flag == 0, confere([razao.min(), razao.max()], [0.64, 0.64], 4),
     round(r.nfev_ls / r.nit) == 24]
print("  tabela k = 0..8: %s | 74 it.: %s | n_g = 75: %s | ||g|| < 1e-6: %s | razão 0.64: %s | cerca de 24 por pesquisa: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------ pesquisa em linha tipo fminbnd
rb = steepest_descent(f, grad, x0, tolg, 1000, ls="fminbnd")
print("\nCom a pesquisa do tipo fminbnd em [0, 1] (TolX = 1e-4): %d iterações, n_g = %d, n_f = %d; %.1f avaliações por pesquisa"
      % (rb.nit, rb.ngev, rb.nfev, rb.nfev_ls / rb.nit))
c = [rb.nit == 74, rb.ngev == 75, rb.nfev_ls / rb.nit == 6]
print("  74 it.: %s | n_g = 75: %s | 6 avaliações por pesquisa: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ gradiente numérico
rn = steepest_descent(f, None, x0, tolg, 1000)
print("\nCom gradiente numérico (grad = None: grad_fd, 2n = 4 avaliações de f por gradiente):")
print("%d iterações, n_g = %d, n_f = %d = %d (pesquisas) + %d (iterandos) + 4 x %d (gradientes)"
      % (rn.nit, rn.ngev, rn.nfev, rn.nfev_ls, rn.nit + 1, rn.nit + 1))
c = [rn.nit == 74, rn.ngev == 0, rn.nfev == rn.nfev_ls + (rn.nit + 1) + 4 * (rn.nit + 1)]
print("  74 it.: %s | n_g = 0: %s | 2n por gradiente em n_f: %s" % simnao(c))
ok = ok and all(c)

rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grosen = lambda x: np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2), 200 * (x[1] - x[0]**2)])
xr = np.array([-1.5, 2.0])
ge = grosen(xr)
print("\nDeck 3.3 — gradiente de Rosenbrock em (-1.5; 2): exato (%g, %g)" % tuple(ge))
erros = []
for h in (1e-2, 1e-4, 1e-6):
    gn, nf = grad_fd(rosen, xr, h)
    erros.append(np.linalg.norm(gn - ge))
    print("  h = %g: (%.6f, %.6f), erro %.1e, %d avaliações de f" % (h, *gn, erros[-1], nf))
gd, nf = grad_fd(rosen, xr)
print("  h por omissão (max(0.01|x_i|, 1e-4)): erro %.1e" % np.linalg.norm(gd - ge))
c = [confere(ge, [-155, -50], 0), confere(erros[1], 6e-6, 6), nf == 4]
print("  grad = (-155, -50): %s | erro 6e-6 com h = 1e-4: %s | 2n = 4 avaliações: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Rosenbrock
print("\nRosenbrock de (-1.5; 2), tolg = 1e-4, kmax = 3000:")
rr = steepest_descent(rosen, grosen, xr, 1e-4, 3000)
print("%s\nx = (%.4f; %.4f), f = %.2e, ||g|| no último gradiente = %.2e; n_g = %d, n_f = %d"
      % (rr.message, *rr.x, rr.fx, rr.history[-2, 4], rr.ngev, rr.nfev))
c = [rr.nit == 3000, rr.flag == 1, confere(rr.fx, 2.4e-4, 5), confere(rr.x, [0.98, 0.97], 2)]
print("  3000 it.: %s | não converge: %s | f = 2.4e-4: %s | x ~ (0.98; 0.97): %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.3.1: método do gradiente

Uma iteração à mão: g0 = (9, 9), d0 = (-9, -9)
alpha_0 = 0.2000 (exato: 162/810 = 0.2), x1 = (7.2000; -0.8000), f(x1) = 28.8000 (era 45)
g1 = (7.2000; -7.2000), g1^T d0 = 4.4e-07 (ortogonais)
  alpha_0 = 0.2: sim | x1: sim | f = 28.8: sim | g1: sim | g1^T d0 = 0: sim

Ziguezague: f(x) = (x1^2 + 9 x2^2)/2, x0 = (9, 1), tolg = 1e-06, pesquisa em linha de Brent (tol 1e-10)
   k                    x_k       f(x_k)    ||g_k|| alpha_k-1
   0  (   9.0000;    1.0000)     45.00000    12.7279       nan
   1  (   7.2000;   -0.8000)     28.80000    10.1823    0.2000
   2  (   5.7600;    0.6400)     18.43200     8.1459    0.2000
   3  (   4.6080;   -0.5120)     11.79648     6.5167    0.2000
   4  (   3.6864;    0.4096)      7.54975     5.2134    0.2000
   5  (   2.9491;   -0.3277)      4.83184     4.1707    0.2000
   6  (   2.3593;    0.2621)      3.09238     3.3365    0.2000
   7  (   1.8874;   -0.2097)      1.97912     2.6692    0.2000
   8  (   1.509

atingiu o número máximo de iterações (kmax = 3000) sem ||g|| <= tolg
x = (0.9844; 0.9690), f = 2.43e-04, ||g|| no último gradiente = 1.99e-02; n_g = 3000, n_f = 60900
  3000 it.: sim | não converge: sim | f = 2.4e-4: sim | x ~ (0.98; 0.97): sim

confere com os slides: sim


## 3.3.2 Método de Newton

Exemplo [`ex03_3_2_newton.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_2_newton/ex03_3_2_newton.py) · funções da UC: [`newton_nd.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_2_newton/newton_nd.py)

```text
Reproduz os exemplos do deck 3.3.2 (método de Newton).

«Podemos usar a curvatura?»: f(x) = (x1^2 + 9 x2^2)/2, x0 = (9, 1):
    H d = -g dá d0 = (-9, -1) e x1 = (0, 0) -- uma iteração.
«Rosenbrock: muito rápido, mas não monótono»: Newton puro de (-1.5; 2),
    tabela k = 0..7 (f sobe para 751 e para 40), 7 iterações, f < 1e-16.
«Newton procura g = 0»: Himmelblau de (0, 0), lambda(H0) = (-42; -26);
    o puro vai em 4 iterações para o máximo local (-0.27; -0.92), f = 181.6;
    o amortecido deteta g^T d >= 0, usa -g e chega a (3, 2) em 5 iterações.
«Newton amortecido»: Rosenbrock, 14 iterações (monótono); os últimos
    minimizantes da linha são 0.91; 0.98; 1.00.
Deck 3.3.3 (quadro de consulta): amortecido com ||g|| < 1e-4: 14 it.,
    n_g = 15, n_H = 14.

Paragem ||g|| <= tolg = 1e-8 (a das figuras), salvo indicação.
Imprime as tabelas, os valores finais e as contagens, e no fim compara com
os slides (vírgula decimal nos slides, ponto aqui).
```

In [8]:
import numpy as np

from newton_nd import newton_nd, DIRECOES


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def confere_sig(v, s, m=3):
    """v confere com s mostrado com m algarismos significativos?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    e = np.floor(np.log10(np.abs(s)))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (e - m + 1) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.3.2: método de Newton")

# ------------------------------------------------ quadrática
A = np.diag([1.0, 9.0])
quad = lambda x: 0.5 * (x[0]**2 + 9 * x[1]**2)     # = x^T A x / 2
gq = lambda x: np.array([x[0], 9 * x[1]])
Hq = lambda x: A
x0 = np.array([9.0, 1.0])
d0 = -np.linalg.solve(A, gq(x0))
r = newton_nd(quad, gq, Hq, x0)
print("\nQuadrática (x1^2 + 9 x2^2)/2 de (9, 1): gradiente -g0 = (%g, %g); Newton H d = -g: d0 = (%g, %g)"
      % (*(-gq(x0)), *d0))
print("x1 = (%g, %g): %d iteração; n_g = %d, n_H = %d" % (*r.x, r.nit, r.ngev, r.nhev))
c = [np.allclose(d0, [-9, -1]), np.allclose(r.x, 0), r.nit == 1]
print("  d0 = (-9, -1): %s | x1 = (0, 0): %s | 1 it.: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Rosenbrock, puro
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grosen = lambda x: np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2), 200 * (x[1] - x[0]**2)])
Hrosen = lambda x: np.array([[2 - 400 * (x[1] - 3 * x[0]**2), -400 * x[0]], [-400 * x[0], 200]])
xr = np.array([-1.5, 2.0])
print("\nRosenbrock de (-1.5; 2), Newton puro (tolg = 1e-8):")
r = newton_nd(rosen, grosen, Hrosen, xr, 1e-8, 50, verbose=True)
H = r.history
print("%s; n_g = %d, n_H = %d, n_f = %d (f só nos iterandos)" % (r.message, r.ngev, r.nhev, r.nfev))
subidas = [(int(H[k, 0]), H[k, 3]) for k in range(1, H.shape[0]) if H[k, 3] > H[k - 1, 3]]
print("f sobe em: " + ", ".join("k = %d (f = %.2f)" % s for s in subidas))
Tx = [[-1.5000, 2.0000], [-1.4510, 2.1029], [0.2044, -2.6986], [0.2059, 0.0424],
      [0.9997, 0.3692], [0.9997, 0.9993], [1.0000, 1.0000], [1.0000, 1.0000]]
Tf = [12.500000, 6.007882, 751.610005, 0.630622, 39.701728]
Tf56 = [1.09e-7, 1.20e-12]
Tg = [1.63e2, 6.31e0, 5.92e2, 1.59e0, 2.82e2, 6.61e-4, 4.89e-5]
c = [H.shape[0] == 8 and confere(H[:, 1:3], Tx, 4) and confere(H[:5, 3], Tf, 6)
     and confere_sig(H[5:7, 3], Tf56) and H[7, 3] == 0 and confere_sig(H[:7, 4], Tg) and H[7, 4] == 0,
     r.nit == 7, r.fx < 1e-16, [s[0] for s in subidas] == [2, 4]]
print("  tabela k = 0..7: %s | 7 it.: %s | f < 1e-16: %s | não monótono (sobe em k = 2 e 4): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Rosenbrock, amortecido
print("\nRosenbrock de (-1.5; 2), Newton amortecido (salvaguarda g^T d >= 0 -> -g; pesquisa em linha de Brent):")
r = newton_nd(rosen, grosen, Hrosen, xr, 1e-8, 50, damped=True, verbose=True)
H = r.history
mono = bool(np.all(np.diff(H[:, 3]) <= 0))
print("%s; n_g = %d, n_H = %d, n_f = %d (%d nas pesquisas em linha)" % (r.message, r.ngev, r.nhev, r.nfev, r.nfev_ls))
print("monótono: %s; últimos minimizantes da linha: %.2f; %.2f; %.2f" % ("sim" if mono else "não", *H[-3:, 5]))
c = [r.nit == 14, mono, confere(H[-3:, 5], [0.91, 0.98, 1.00], 2)]
print("  14 it.: %s | monótono: %s | alpha = 0.91; 0.98; 1.00: %s" % simnao(c))
ok = ok and all(c)
r4 = newton_nd(rosen, grosen, Hrosen, xr, 1e-4, 50, damped=True)
print("Com tolg = 1e-4 (quadro de consulta do 3.3.3): %d it., n_g = %d, n_H = %d" % (r4.nit, r4.ngev, r4.nhev))
c = [r4.nit == 14, r4.ngev == 15, r4.nhev == 14]
print("  14 it.: %s | n_g = 15: %s | n_H = 14: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Himmelblau
himmel = lambda x: (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
ghimmel = lambda x: np.array([4 * x[0] * (x[0]**2 + x[1] - 11) + 2 * (x[0] + x[1]**2 - 7),
                              2 * (x[0]**2 + x[1] - 11) + 4 * x[1] * (x[0] + x[1]**2 - 7)])
Hhimmel = lambda x: np.array([[12 * x[0]**2 + 4 * x[1] - 42, 4 * x[0] + 4 * x[1]],
                              [4 * x[0] + 4 * x[1], 4 * x[0] + 12 * x[1]**2 - 26]])
xh = np.array([0.0, 0.0])
lam0 = np.linalg.eigvalsh(Hhimmel(xh))
print("\nHimmelblau de (0, 0): lambda(H0) = (%g; %g) -- definida negativa" % tuple(lam0))
rp = newton_nd(himmel, ghimmel, Hhimmel, xh, 1e-8, 50)
lamx = np.linalg.eigvalsh(Hhimmel(rp.x))
print("Puro: %d it. -> x = (%.4f; %.4f), f = %.4f, lambda(H) = (%.1f; %.1f) < 0: máximo local"
      % (rp.nit, *rp.x, rp.fx, *lamx))
rd = newton_nd(himmel, ghimmel, Hhimmel, xh, 1e-8, 50, damped=True)
print("Amortecido: direção na 1.ª iteração: %s; %d it. -> x = (%.4f; %.4f), f = %.1e"
      % (DIRECOES[int(rd.history[1, 7])], rd.nit, *rd.x, rd.fx))
c = [np.allclose(lam0, [-42, -26]), rp.nit == 4, confere(rp.x, [-0.27, -0.92], 2), confere(rp.fx, 181.6, 1),
     bool(np.all(lamx < 0)), rd.history[1, 7] == 2, rd.nit == 5, np.allclose(rd.x, [3, 2])]
print("  lambda(H0) = (-42; -26): %s | puro 4 it.: %s | (-0.27; -0.92): %s | f = 181.6: %s | máximo: %s"
      " | amortecido usa -g: %s | 5 it.: %s | (3, 2): %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.3.2: método de Newton

Quadrática (x1^2 + 9 x2^2)/2 de (9, 1): gradiente -g0 = (-9, -9); Newton H d = -g: d0 = (-9, -1)
x1 = (0, 0): 1 iteração; n_g = 2, n_H = 1
  d0 = (-9, -1): sim | x1 = (0, 0): sim | 1 it.: sim

Rosenbrock de (-1.5; 2), Newton puro (tolg = 1e-8):
   k                      x_k           f(x_k)     ||g_k||     alpha  direção
   0  (  -1.5000;    2.0000)     1.250000e+01    1.63e+02       nan  
   1  (  -1.4510;    2.1029)     6.007882e+00    6.31e+00    1.0000  Newton
   2  (   0.2044;   -2.6986)     7.516100e+02    5.92e+02    1.0000  Newton
   3  (   0.2059;    0.0424)     6.306223e-01    1.59e+00    1.0000  Newton
   4  (   0.9997;    0.3692)     3.970173e+01    2.82e+02    1.0000  Newton
   5  (   0.9997;    0.9993)     1.093364e-07    6.61e-04    1.0000  Newton
   6  (   1.0000;    1.0000)     1.195444e-12    4.89e-05    1.0000  Newton
   7  (   1.0000;    1.0000)     0.000000e+00    0.00e+00    1.0000  Newton
||g|| <= tolg em 7 iterações; n_

## 3.3.3 Método dos gradientes conjugados (Fletcher–Reeves)

Exemplo [`ex03_3_3_conjugate_gradients.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/ex03_3_3_conjugate_gradients.py) · funções da UC: [`conj_grad.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/conj_grad.py)

```text
Reproduz os exemplos do deck 3.3.3 (gradientes conjugados, Fletcher-Reeves).

Quadrática f(x) = (x1^2 + 9 x2^2)/2 de x0 = (9, 1), pesquisa em linha exata:
    o 1.º passo é o do gradiente, x1 = (7.2; -0.8); beta_1 = 103.68/162 = 0.64,
    d1 = (-12.96; 1.44), d0^T A d1 = 0 (A-conjugadas), alpha_1 = 5/9,
    x2 = (0, 0): 2 iterações, n_g = 3.
Rosenbrock de (-1.5; 2), ||g|| < 1e-4: com reinício a cada n = 2, 36 it.
    (n_g = 37); sem reinício, 102 it.
«No computador»: com uma pesquisa do tipo fminbnd de tolerância por
    omissão, FR no Rosenbrock pode não convergir (informativo).

Imprime as tabelas, os valores finais e as contagens, e no fim compara com
os slides (vírgula decimal nos slides, ponto aqui).
```

In [9]:
import numpy as np

from conj_grad import conj_grad


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.3.3: gradientes conjugados (Fletcher-Reeves)")

# ------------------------------------------------ quadrática
A = np.diag([1.0, 9.0])
quad = lambda x: 0.5 * (x[0]**2 + 9 * x[1]**2)     # = x^T A x / 2
gq = lambda x: np.array([x[0], 9 * x[1]])
x0 = np.array([9.0, 1.0])
print("\nQuadrática (x1^2 + 9 x2^2)/2 de (9, 1), tolg = 1e-6, pesquisa em linha de Brent (tol 1e-10):")
r = conj_grad(quad, gq, x0, 1e-6, 100, verbose=True)
H = r.history                          # [k, x1, x2, f, ||g||, alpha, beta, d1, d2]
g0, g1 = gq(H[0, 1:3]), gq(H[1, 1:3])
d0, d1 = H[0, 7:9], H[1, 7:9]
print("||g1||^2/||g0||^2 = %.2f/%.0f = %.4f; d0^T A d1 = %.1e; alpha_1 = %.6f (5/9 = %.6f)"
      % (g1 @ g1, g0 @ g0, H[1, 6], d0 @ A @ d1, H[2, 5], 5 / 9))
print("%s; n_g = %d, n_f = %d" % (r.message, r.ngev, r.nfev))
c = [confere(H[1, 1:3], [7.2, -0.8], 4), confere(g1 @ g1, 103.68, 2), confere(H[1, 6], 0.64, 4),
     confere(d1, [-12.96, 1.44], 2), abs(d0 @ A @ d1) < 1e-4, confere(H[2, 5], 5 / 9, 6),
     confere(r.x, [0, 0], 6), r.nit == 2, r.ngev == 3]
print("  x1: %s | ||g1||^2 = 103.68: %s | beta_1 = 0.64: %s | d1: %s | d0'Ad1 = 0: %s | alpha_1 = 5/9: %s"
      " | x2 = (0, 0): %s | 2 it.: %s | n_g = 3: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Rosenbrock
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grosen = lambda x: np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2), 200 * (x[1] - x[0]**2)])
xr = np.array([-1.5, 2.0])
print("\nRosenbrock de (-1.5; 2), tolg = 1e-4:")
rc = conj_grad(rosen, grosen, xr, 1e-4, 500)                # reinício a cada n = 2
rs = conj_grad(rosen, grosen, xr, 1e-4, 500, restart=0)     # sem reinício periódico
print("FR com reinício a cada n = 2: %d it., n_g = %d, n_f = %d, f = %.1e (%d reinícios)"
      % (rc.nit, rc.ngev, rc.nfev, rc.fx, rc.nrestart))
print("FR sem reinício periódico:    %d it., n_g = %d, n_f = %d, f = %.1e (%d reinícios por g^T d >= 0)"
      % (rs.nit, rs.ngev, rs.nfev, rs.fx, rs.nrestart))
c = [rc.nit == 36, rc.ngev == 37, rs.nit == 102, rc.flag == 0 and rs.flag == 0]
print("  36 it. com reinício: %s | n_g = 37: %s | 102 it. sem: %s | convergem: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------- pesquisa tipo fminbnd (informativo)
rb = conj_grad(rosen, grosen, xr, 1e-4, 500, ls="fminbnd")
print("\nCom a pesquisa do tipo fminbnd (TolX = 1e-4) e reinício a cada n: %s; f = %.2g"
      % (rb.message, rb.fx))
print("  (o slide diz que «pode não convergir»: %s)" % ("não convergiu" if rb.flag else "convergiu"))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.3.3: gradientes conjugados (Fletcher-Reeves)

Quadrática (x1^2 + 9 x2^2)/2 de (9, 1), tolg = 1e-6, pesquisa em linha de Brent (tol 1e-10):
   k                    x_k         f(x_k)     ||g_k||     alpha     beta                    d_k
   0  (   9.0000;    1.0000)      45.000000    1.27e+01       nan      nan  (  -9.0000;   -9.0000)
   1  (   7.2000;   -0.8000)      28.800000    1.02e+01    0.2000   0.6400  ( -12.9600;    1.4400)
   2  (  -0.0000;    0.0000)       0.000000    7.36e-13    0.5556      nan  (      nan;       nan)
||g1||^2/||g0||^2 = 103.68/162 = 0.6400; d0^T A d1 = -7.9e-07; alpha_1 = 0.555556 (5/9 = 0.555556)
||g|| <= tolg em 2 iterações; n_g = 3, n_f = 47
  x1: sim | ||g1||^2 = 103.68: sim | beta_1 = 0.64: sim | d1: sim | d0'Ad1 = 0: sim | alpha_1 = 5/9: sim | x2 = (0, 0): sim | 2 it.: sim | n_g = 3: sim

Rosenbrock de (-1.5; 2), tolg = 1e-4:
FR com reinício a cada n = 2: 36 it., n_g = 37, n_f = 775, f = 1.5e-12 (17 reinícios)
FR sem reinício periódi

## 3.3.3 Os métodos do capítulo 3 no mesmo problema

Exemplo [`ex03_3_comparison.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_comparison/ex03_3_comparison.py) · funções da UC: [`box_evo.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_4_box/box_evo.py), [`conj_grad.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/conj_grad.py), [`fg.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_comparison/fg.py), [`hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/hooke_jeeves.py), [`local_random_search.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_1_random_search/local_random_search.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py), [`newton_nd.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_2_newton/newton_nd.py), [`steepest_descent.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_1_steepest_descent/steepest_descent.py)

```text
Reproduz a tabela «Os métodos estudados no mesmo problema, com o custo em
avaliações» do fim do capítulo 3 (deck 3.3.3), com as funções da UC.

Regras (folha de convenções da UC):
* mesmo problema (Rosenbrock, fg.py), mesmo ponto inicial x0 = (-1.5; 2),
  mesmo critério: custo até à PRIMEIRA avaliação de f com f < 1e-4 (f nos
  iterandos dos métodos com derivadas também conta em n_f);
* custo em avaliações contadas à parte, n_f, n_g, n_H; equivalente em f com
  derivadas centrais (n = 2): n_f + 4 n_g + 4 n_H (a Hessiana, 2n^2+1 = 9
  pontos, reaproveita os 4 do gradiente e f(x_k));
* métodos de 3.3 com a pesquisa em linha precisa (Brent, tol 1e-10);
* pesquisa aleatória localizada (estocástica): 30 corridas, sementes 0..29
  (numpy.random.default_rng), mediana [quartis].

Valores dos slides (n_f / n_g / n_H / equiv.):
  aleatória localizada 975 [867; 1052]; Nelder-Mead 166; Box 13 445;
  Hooke-Jeeves 430; gradiente 60 900 / 3000 / - / 72 900 (não atinge
  f < 1e-4 em 3000 it.); FR (reinício n) 718 / 34 / - / 854;
  Newton puro 6 / 5 / 5 / 46; Newton amortecido 281 / 12 / 12 / 377.

Usa as funções das pastas 03_2_*, 03_3_* e common/ (postas no sys.path por
uc_setup).
```

In [10]:
import numpy as np

from box_evo import box_evo
from conj_grad import conj_grad
from fg import fg
from hooke_jeeves import hooke_jeeves
from local_random_search import local_random_search
from nelder_mead import nelder_mead
from newton_nd import newton_nd
from steepest_descent import steepest_descent


class Contador:
    """Conta n_f, n_g, n_H e guarda (n_f, n_g, n_H, f) em cada avaliação de f."""
    def __init__(self):
        self.nf = self.ng = self.nH = 0
        self.trace = []

    def F(self, x):
        v = fg(x); self.nf += 1
        self.trace.append((self.nf, self.ng, self.nH, v))
        return v

    def G(self, x):
        self.ng += 1
        return fg(x, 2)[1]

    def H(self, x):
        self.nH += 1
        return fg(x, 3)[2]

    def primeira(self, tol):
        """(n_f, n_g, n_H) na primeira avaliação com f < tol; None se nunca."""
        for nf, ng, nH, v in self.trace:
            if v < tol:
                return nf, ng, nH
        return None


def equiv(nf, ng, nH, n=2):
    """custo equivalente em f com derivadas centrais: n_f + 2n n_g + (2n^2 - 2n) n_H."""
    return nf + 2 * n * ng + (2 * n * n - 2 * n) * nH


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


X0 = np.array([-1.5, 2.0])
FTOL = 1e-4
ok = True
print("Otimização — capítulo 3: os métodos estudados no mesmo problema (Rosenbrock de (-1.5; 2))")
print("Critério: primeira avaliação com f < %g; custo equivalente em f = n_f + 4 n_g + 4 n_H" % FTOL)

metodos = [
    ("Nelder-Mead", 0, lambda C: nelder_mead(C.F, [X0, X0 + [0.5, 0], X0 + [0, 0.5]], 1e-8, 1e-10, 1000)),
    ("Box (Delta_0 = 1)", 0, lambda C: box_evo(C.F, X0, [1, 1], 1e-6, 10000)),
    ("Hooke-Jeeves (P_0 = 0.5)", 0, lambda C: hooke_jeeves(C.F, X0, 2, 0.5, 1e-6, 100000)),
    ("Gradiente", 1, lambda C: steepest_descent(C.F, C.G, X0, 1e-8, 3000)),
    ("Grad. conj. (FR, reinício n)", 1, lambda C: conj_grad(C.F, C.G, X0, 1e-8, 3000)),
    ("Newton puro", 2, lambda C: newton_nd(C.F, C.G, C.H, X0, 1e-8, 200)),
    ("Newton amortecido", 2, lambda C: newton_nd(C.F, C.G, C.H, X0, 1e-8, 200, damped=True, modify=True)),
]
slides = {"Nelder-Mead": (166, 0, 0, 166), "Box (Delta_0 = 1)": (13445, 0, 0, 13445),
          "Hooke-Jeeves (P_0 = 0.5)": (430, 0, 0, 430), "Gradiente": (60900, 3000, 0, 72900),
          "Grad. conj. (FR, reinício n)": (718, 34, 0, 854), "Newton puro": (6, 5, 5, 46),
          "Newton amortecido": (281, 12, 12, 377)}

# ------------------------------------------ pesquisa aleatória localizada
nhit = []
for s in range(30):
    C = Contador()
    local_random_search(C.F, X0, 1.0, 20, 0.9, 1e-8, 5000, s=s)
    hit = C.primeira(FTOL)
    nhit.append(hit[0] if hit else np.inf)
nhit = np.array(nhit)
q1, med, q3 = np.percentile(nhit[np.isfinite(nhit)], [25, 50, 75])
nfalhas = int(np.sum(~np.isfinite(nhit)))

print("\n%-30s %6s %18s %6s %5s %8s" % ("Método", "ordem", "n_f", "n_g", "n_H", "equiv."))
print("%-30s %6d %18s %6s %5s %8.0f" % ("Aleatória localizada (30x)", 0, "%.0f [%.0f; %.0f]" % (med, q1, q3),
                                         "--", "--", med))
linhas_ok = [round(med) == 975 and round(q1) == 867 and round(q3) == 1052 and nfalhas == 0]   # q3 = 1051.75

resultados = {}
for nome, ordem, corre in metodos:
    C = Contador()
    r = corre(C)
    hit = C.primeira(FTOL)
    # as contagens do contador coincidem com as da própria função?
    proprias = (r.nfev, getattr(r, "ngev", 0), getattr(r, "nhev", 0))
    if proprias != (C.nf, C.ng, C.nH):
        print("  AVISO: %s conta (%d, %d, %d), o contador (%d, %d, %d)" % (nome, *proprias, C.nf, C.ng, C.nH))
        ok = False
    if hit is None:                                 # não atinge: custo total da corrida
        nf, ng, nH = C.nf, C.ng, C.nH
        marca = "‡"
    else:
        nf, ng, nH = hit
        marca = ""
    eq = equiv(nf, ng, nH)
    resultados[nome] = (nf, ng, nH, eq, r)
    print("%-30s %6d %18d %6s %5s %8d" % (nome + marca, ordem, nf, ng if ng else "--", nH if nH else "--", eq))
    linhas_ok.append((nf, ng, nH, eq) == slides[nome])
print("‡ não atinge f < 1e-4 em 3000 iterações (f = %.1e); custo total da corrida"
      % resultados["Gradiente"][4].fx)

nd = resultados["Newton amortecido"]; fr = resultados["Grad. conj. (FR, reinício n)"]
print("\nDeck 3.3.3: até f < 1e-4, Newton amortecido n_g = n_H = %d, n_f = %d na pesquisa em linha (%d com f nos iterandos);"
      % (nd[1], nd[0] - nd[1], nd[0]))
print("            FR n_g = %d, n_f = %d na pesquisa em linha (%d)."
      % (fr[1], fr[0] - fr[1], fr[0]))
print("            (a primeira f < 1e-4 surge numa pesquisa em linha; até aí os iterandos avaliados são tantos quantos os gradientes)")
c2 = [nd[1] == 12 and nd[0] - nd[1] == 269, fr[1] == 34 and fr[0] - fr[1] == 684]
print("  Newton: 269 + 12 = 281: %s | FR: 684 + 34 = 718: %s" % simnao(c2))

print("\n  aleatória 975 [867; 1052], 30/30: %s | NM 166: %s | Box 13 445: %s | HJ 430: %s | gradiente 72 900: %s"
      " | FR 854: %s | Newton puro 46: %s | amortecido 377: %s" % simnao(linhas_ok))
ok = ok and all(linhas_ok) and all(c2)

# ------------------------------------------------ o mesmo com o SciPy (informativo)
try:
    from scipy.optimize import minimize
    print("\nO mesmo com o SciPy (informativo: critério e pesquisa em linha do SciPy, números diferentes):")
    f1 = lambda x: fg(x)
    g1 = lambda x: fg(x, 2)[1]
    h1 = lambda x: fg(x, 3)[2]
    for metodo, kw in (("Nelder-Mead", {}), ("CG", {"jac": g1}), ("BFGS", {"jac": g1}),
                       ("Newton-CG", {"jac": g1, "hess": h1})):
        r = minimize(f1, X0, method=metodo, tol=1e-8, **kw)
        print("  %-12s f = %.1e  nfev = %4d  njev = %4d  nhev = %4d"
              % (metodo, r.fun, r.nfev, getattr(r, "njev", 0), getattr(r, "nhev", 0)))
except ImportError:
    print("\n(SciPy não instalado: parte informativa omitida)")

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — capítulo 3: os métodos estudados no mesmo problema (Rosenbrock de (-1.5; 2))
Critério: primeira avaliação com f < 0.0001; custo equivalente em f = n_f + 4 n_g + 4 n_H



Método                          ordem                n_f    n_g   n_H   equiv.
Aleatória localizada (30x)          0    975 [867; 1052]     --    --      975
Nelder-Mead                         0                166     --    --      166
Box (Delta_0 = 1)                   0              13445     --    --    13445
Hooke-Jeeves (P_0 = 0.5)            0                430     --    --      430


Gradiente‡                          1              60900   3000    --    72900
Grad. conj. (FR, reinício n)        1                718     34    --      854
Newton puro                         2                  6      5     5       46
Newton amortecido                   2                281     12    12      377
‡ não atinge f < 1e-4 em 3000 iterações (f = 2.4e-04); custo total da corrida

Deck 3.3.3: até f < 1e-4, Newton amortecido n_g = n_H = 12, n_f = 269 na pesquisa em linha (281 com f nos iterandos);
            FR n_g = 34, n_f = 684 na pesquisa em linha (718).
            (a primeira f < 1e-4 surge numa pesquisa em linha; até aí os iterandos avaliados são tantos quantos os gradientes)
  Newton: 269 + 12 = 281: sim | FR: 684 + 34 = 718: sim

  aleatória 975 [867; 1052], 30/30: sim | NM 166: sim | Box 13 445: sim | HJ 430: sim | gradiente 72 900: sim | FR 854: sim | Newton puro 46: sim | amortecido 377: sim



O mesmo com o SciPy (informativo: critério e pesquisa em linha do SciPy, números diferentes):
  Nelder-Mead  f = 2.3e-18  nfev =  237  njev =    0  nhev =    0
  CG           f = 8.5e-25  nfev =   81  njev =   79  nhev =    0
  BFGS         f = 9.2e-23  nfev =   45  njev =   45  nhev =    0
  Newton-CG    f = 6.6e-23  nfev =  211  njev =  211  nhev =  193

confere com os slides: sim


## Caderno da comparação do capítulo 3

O caderno autónomo [`cap3_comparacao.ipynb`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_comparison/cap3_comparacao.ipynb) faz a mesma comparação com gráficos. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_comparison/cap3_comparacao.ipynb)